# Sailor Shift — Data Preparation for Tableau

Produces flat CSVs from the two source files (`mc1_nodes.csv`, `mc1_edges.csv`) for the influence and collaborator dashboards.

**Edge direction convention:**
- `PerformerOf / ComposerOf / LyricistOf / ProducerOf`: source=Person → target=Work
- Influence edges (`CoverOf`, `DirectlySamples`, `InStyleOf`, `InterpolatesFrom`, `LyricalReferenceTo`): source=Work → target=Work, meaning **source was influenced by target**
- `RecordedBy`: source=Work → target=RecordLabel
- `MemberOf`: source=Person → target=MusicalGroup

In [1]:
import pandas as pd
import os

DATA_DIR = "../mc1_csv/"
OUT_DIR = "./sailor_csvs/"
os.makedirs(OUT_DIR, exist_ok=True)

nodes = pd.read_csv(DATA_DIR + "mc1_nodes.csv")
edges = pd.read_csv(DATA_DIR + "mc1_edges.csv")

print(f"Nodes: {len(nodes):,} | Edges: {len(edges):,}")
nodes.head(3)

Nodes: 17,412 | Edges: 37,857


,Node Type,name,single,release_date,genre,notable,id,written_date,stage_name,notoriety_date
0,Song,Breaking These Chains,True,2017.0,Oceanus Folk,True,0,NaN,NaN,NaN
1,Person,Carlos Duffy,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN
2,Person,Min Qin,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN


In [2]:
# ── Constants ──────────────────────────────────────────────────────────────
SAILOR_ID = 17255

AUTHORSHIP_EDGES = {"PerformerOf", "ComposerOf", "LyricistOf", "ProducerOf"}
INFLUENCE_EDGES  = {"CoverOf", "DirectlySamples", "InStyleOf", "InterpolatesFrom", "LyricalReferenceTo"}
WORK_TYPES       = {"Song", "Album"}

node_lookup = nodes.set_index("id")
person_ids  = set(nodes[nodes["Node Type"] == "Person"]["id"])

def pipe_join(series):
    """Aggregate helper: join unique non-null values with |."""
    return "|".join(series.dropna().astype(str).unique())

print(f"Sailor Shift node: {node_lookup.loc[SAILOR_ID, 'name']}")

Sailor Shift node: Sailor Shift


## 1. Sailor's Works
All Songs/Albums where Sailor has an authorship credit.

In [3]:
sailor_auth = edges[
    (edges["Edge Type"].isin(AUTHORSHIP_EDGES)) &
    (edges["source"] == SAILOR_ID)
].copy()

sailor_work_ids = set(sailor_auth["target"].unique())

sailor_works = (
    node_lookup
    .loc[node_lookup.index.isin(sailor_work_ids)]
    .reset_index()
    .rename(columns={"id": "work_id", "Node Type": "work_type", "name": "work_name",
                     "release_date": "release_year", "genre": "genre", "notable": "notable"})
    [["work_id", "work_type", "work_name", "release_year", "genre", "notable"]]
)

role_agg = (
    sailor_auth
    .groupby("target", as_index=False)
    .agg(sailor_roles=("Edge Type", pipe_join))
    .rename(columns={"target": "work_id"})
)
sailor_works = sailor_works.merge(role_agg, on="work_id", how="left")

sailor_works.to_csv(OUT_DIR + "sailor_works.csv", index=False)
print(f"Sailor's works: {len(sailor_works)}")
sailor_works.head()

Sailor's works: 38


,work_id,work_type,work_name,release_year,genre,notable,sailor_roles
0,16961,Album,Neon Heartbeat,2031.0,Synthwave,False,LyricistOf
1,16989,Album,Ballads for the End of Time,2033.0,Oceanus Folk,True,LyricistOf
2,16999,Album,Melancholy Circuitry,2033.0,Americana,True,LyricistOf
3,17047,Album,Drifting Between the Stars and the Sea,2034.0,Oceanus Folk,True,LyricistOf
4,17048,Album,Artificial Sunsets,2035.0,Oceanus Folk,True,LyricistOf


## 2. Works that Influenced Sailor

Influence edge direction: `source → target` means source was influenced by target.  
So edges where **source is one of Sailor's works** → the target is the influencing work.

Then resolve: who created those influencing works?

In [4]:
inbound = edges[
    (edges["Edge Type"].isin(INFLUENCE_EDGES)) &
    (edges["source"].isin(sailor_work_ids))
].copy().rename(columns={
    "source": "sailor_work_id",
    "target": "influencing_work_id",
    "Edge Type": "influence_type"
})

# Sailor's work metadata
inbound = inbound.merge(
    sailor_works[["work_id", "work_name", "release_year", "genre", "notable"]]
    .rename(columns={"work_id": "sailor_work_id", "work_name": "sailor_work_name",
                     "release_year": "sailor_work_year", "genre": "sailor_work_genre",
                     "notable": "sailor_work_notable"}),
    on="sailor_work_id", how="left"
)

# Influencing work metadata
influencing_meta = (
    node_lookup
    .loc[node_lookup.index.isin(inbound["influencing_work_id"])]
    .reset_index()[["id", "name", "release_date", "genre"]]
    .rename(columns={"id": "influencing_work_id", "name": "influencing_work_name",
                     "release_date": "influencing_work_year", "genre": "influencing_work_genre"})
)
inbound = inbound.merge(influencing_meta, on="influencing_work_id", how="left")

# Who created those influencing works?
creator_edges = edges[
    (edges["Edge Type"].isin(AUTHORSHIP_EDGES)) &
    (edges["target"].isin(inbound["influencing_work_id"]))
].copy().rename(columns={"source": "creator_id", "target": "influencing_work_id", "Edge Type": "creator_role"})

creator_edges = creator_edges.merge(
    node_lookup[["name"]].reset_index().rename(columns={"id": "creator_id", "name": "creator_name"}),
    on="creator_id", how="left"
)

creator_agg = (
    creator_edges
    .groupby("influencing_work_id", as_index=False)
    .agg(
        influencing_creators=("creator_name", pipe_join),
        influencing_creator_ids=("creator_id", pipe_join)
    )
)
inbound = inbound.merge(creator_agg, on="influencing_work_id", how="left")

inbound.to_csv(OUT_DIR + "sailor_influenced_by.csv", index=False)
print(f"Inbound influence edges: {len(inbound)}")
inbound.head(3)

Inbound influence edges: 26


,influence_type,sailor_work_id,influencing_work_id,key,sailor_work_name,sailor_work_year,sailor_work_genre,sailor_work_notable,influencing_work_name,influencing_work_year,influencing_work_genre,influencing_creators,influencing_creator_ids
0,CoverOf,16999,14893,0,Melancholy Circuitry,2033.0,Americana,True,Twilight's Threshold,2007.0,Synthwave,Ming Long|Yong Lu|Kimberly Estrada|Fang Ding|P...,1335|1336|14894|14895|16910
1,InStyleOf,17049,14107,0,Electric Reverie,2038.0,Oceanus Folk,True,Folklore's Heartbeat,2020.0,Blues Rock,Brooke Olson|Tao Hu,12948|14108
2,InterpolatesFrom,17049,7026,0,Electric Reverie,2038.0,Oceanus Folk,True,Reflejo Interior,1983.0,Americana,Sheryl Roman|John Graves|Jason Aguirre|Matthew...,7027|7028|7029|7030


## 2b. Influence by Person — Long Format

One row per (sailor_work, influencing_person) — better for Tableau bump charts and person-level filters.

In [5]:
inbound_long = inbound.merge(
    creator_edges[["influencing_work_id", "creator_id", "creator_name", "creator_role"]],
    on="influencing_work_id", how="left"
).drop(columns=["influencing_creators", "influencing_creator_ids"])

# Filter to Person nodes only
inbound_long = inbound_long[inbound_long["creator_id"].isin(person_ids)]

inbound_long.to_csv(OUT_DIR + "sailor_influenced_by_long.csv", index=False)
print(f"Inbound influence (long): {len(inbound_long)} rows")
inbound_long.head(3)

Inbound influence (long): 122 rows


,influence_type,sailor_work_id,influencing_work_id,key,sailor_work_name,sailor_work_year,sailor_work_genre,sailor_work_notable,influencing_work_name,influencing_work_year,influencing_work_genre,creator_id,creator_name,creator_role
0,CoverOf,16999,14893,0,Melancholy Circuitry,2033.0,Americana,True,Twilight's Threshold,2007.0,Synthwave,1335,Ming Long,PerformerOf
1,CoverOf,16999,14893,0,Melancholy Circuitry,2033.0,Americana,True,Twilight's Threshold,2007.0,Synthwave,1335,Ming Long,ProducerOf
2,CoverOf,16999,14893,0,Melancholy Circuitry,2033.0,Americana,True,Twilight's Threshold,2007.0,Synthwave,1336,Yong Lu,LyricistOf


## 3. Works Sailor Influenced (Outbound)

Edges where **target is one of Sailor's works** → source was influenced by Sailor.

In [6]:
outbound = edges[
    (edges["Edge Type"].isin(INFLUENCE_EDGES)) &
    (edges["target"].isin(sailor_work_ids))
].copy().rename(columns={
    "target": "sailor_work_id",
    "source": "influenced_work_id",
    "Edge Type": "influence_type"
})

# Sailor's work metadata
outbound = outbound.merge(
    sailor_works[["work_id", "work_name", "release_year", "genre"]]
    .rename(columns={"work_id": "sailor_work_id", "work_name": "sailor_work_name",
                     "release_year": "sailor_work_year", "genre": "sailor_work_genre"}),
    on="sailor_work_id", how="left"
)

# Influenced work metadata
influenced_meta = (
    node_lookup
    .loc[node_lookup.index.isin(outbound["influenced_work_id"])]
    .reset_index()[["id", "name", "release_date", "genre", "notable"]]
    .rename(columns={"id": "influenced_work_id", "name": "influenced_work_name",
                     "release_date": "influenced_work_year", "genre": "influenced_work_genre",
                     "notable": "influenced_work_notable"})
)
outbound = outbound.merge(influenced_meta, on="influenced_work_id", how="left")

# Who created those influenced works?
influenced_creator_edges = edges[
    (edges["Edge Type"].isin(AUTHORSHIP_EDGES)) &
    (edges["target"].isin(outbound["influenced_work_id"].unique()))
].copy().rename(columns={"source": "creator_id", "target": "influenced_work_id", "Edge Type": "creator_role"})

influenced_creator_edges = influenced_creator_edges.merge(
    node_lookup[["name"]].reset_index().rename(columns={"id": "creator_id", "name": "creator_name"}),
    on="creator_id", how="left"
)

out_creator_agg = (
    influenced_creator_edges
    .groupby("influenced_work_id", as_index=False)
    .agg(influenced_creators=("creator_name", pipe_join))
)
outbound = outbound.merge(out_creator_agg, on="influenced_work_id", how="left")

outbound.to_csv(OUT_DIR + "sailor_influenced_others.csv", index=False)
print(f"Outbound influence edges: {len(outbound)}")
outbound.head(3)

Outbound influence edges: 0


,influence_type,key,sailor_work_id,sailor_work_name,sailor_work_year,sailor_work_genre,influenced_work_name,influenced_work_year,influenced_work_genre,influenced_work_notable,influenced_work_id,influenced_creators


## 4. Sailor's Collaborators

Any person who shares an authorship credit on the same work as Sailor.

In [7]:
collab_edges = edges[
    (edges["Edge Type"].isin(AUTHORSHIP_EDGES)) &
    (edges["target"].isin(sailor_work_ids)) &
    (edges["source"] != SAILOR_ID)
].copy()

collab_edges = collab_edges[collab_edges["source"].isin(person_ids)]
collab_edges = collab_edges.rename(columns={
    "source": "collaborator_id", "target": "work_id", "Edge Type": "collab_role"
})

collab_edges = collab_edges.merge(
    node_lookup[["name"]].reset_index().rename(columns={"id": "collaborator_id", "name": "collaborator_name"}),
    on="collaborator_id", how="left"
).merge(
    sailor_works[["work_id", "work_name", "release_year", "genre", "notable", "work_type"]],
    on="work_id", how="left"
)

collab_edges.to_csv(OUT_DIR + "sailor_collaborators.csv", index=False)
print(f"Collaborator credit rows: {len(collab_edges)}")
print(f"Unique collaborators: {collab_edges['collaborator_id'].nunique()}")
collab_edges.head(3)

Collaborator credit rows: 67
Unique collaborators: 40


,collab_role,collaborator_id,work_id,key,collaborator_name,work_name,release_year,genre,notable,work_type
0,LyricistOf,16958,16961,0,Zane Cruz,Neon Heartbeat,2031.0,Synthwave,False,Album
1,ComposerOf,16959,16961,0,Iris Moon,Neon Heartbeat,2031.0,Synthwave,False,Album
2,ComposerOf,16983,16989,0,Sophie Bennett,Ballads for the End of Time,2033.0,Oceanus Folk,True,Album


## 5. Collaborator Genre Before/After

For each collaborator, find their **first collaboration year with Sailor**, then classify all their works as Before or After.

This shows whether collaborating with Sailor shifted a person's genre output.

In [8]:
first_collab = (
    collab_edges
    .dropna(subset=["release_year"])
    .groupby("collaborator_id", as_index=False)
    .agg(first_collab_year=("release_year", "min"))
)

collaborator_ids = set(collab_edges["collaborator_id"])

# All works by collaborators (across the full graph, not just Sailor's works)
all_cw = edges[
    (edges["Edge Type"].isin(AUTHORSHIP_EDGES)) &
    (edges["source"].isin(collaborator_ids))
].copy().rename(columns={"source": "collaborator_id", "Edge Type": "collab_role"})

# Attach work metadata
work_meta = (
    node_lookup[["name", "release_date", "genre", "Node Type", "notable"]]
    .reset_index()
    .rename(columns={"id": "target", "name": "work_name", "release_date": "release_year",
                     "Node Type": "work_type"})
)
all_cw = all_cw.merge(work_meta, on="target", how="left")
all_cw = all_cw[all_cw["work_type"].isin(WORK_TYPES)].copy()

# Join first collab year and collaborator name
all_cw = all_cw.merge(first_collab, on="collaborator_id", how="left")
all_cw = all_cw.merge(
    node_lookup[["name"]].reset_index().rename(columns={"id": "collaborator_id", "name": "collaborator_name"}),
    on="collaborator_id", how="left"
)

# Flag Before / After first collab with Sailor
all_cw["pre_post"] = "After"
mask = (all_cw["release_year"].notna() & all_cw["first_collab_year"].notna() &
        (all_cw["release_year"] < all_cw["first_collab_year"]))
all_cw.loc[mask, "pre_post"] = "Before"

collab_genre_ba = all_cw[[
    "collaborator_id", "collaborator_name", "first_collab_year",
    "target", "work_name", "release_year", "genre", "notable", "work_type", "collab_role", "pre_post"
]].rename(columns={"target": "work_id"})

collab_genre_ba.to_csv(OUT_DIR + "collaborator_genres_before_after.csv", index=False)
print(f"Collaborator work rows: {len(collab_genre_ba)}")
collab_genre_ba.head(3)

Collaborator work rows: 146


,collaborator_id,collaborator_name,first_collab_year,work_id,work_name,release_year,genre,notable,work_type,collab_role,pre_post
0,16958,Zane Cruz,2031.0,16961,Neon Heartbeat,2031.0,Synthwave,False,Album,LyricistOf,After
1,16958,Zane Cruz,2031.0,16962,Cosmic Lullabies,2035.0,Synthwave,True,Album,LyricistOf,After
2,16959,Iris Moon,2031.0,16961,Neon Heartbeat,2031.0,Synthwave,False,Album,ComposerOf,After


## 6. Summary Check

In [9]:
files = [
    "sailor_works.csv",
    "sailor_influenced_by.csv",
    "sailor_influenced_by_long.csv",
    "sailor_influenced_others.csv",
    "sailor_collaborators.csv",
    "collaborator_genres_before_after.csv",
]

for f in files:
    df = pd.read_csv(OUT_DIR + f)
    print(f"{f}: {len(df):,} rows")
    print(f"  columns: {df.columns.tolist()}")
    print()

sailor_works.csv: 38 rows
  columns: ['work_id', 'work_type', 'work_name', 'release_year', 'genre', 'notable', 'sailor_roles']

sailor_influenced_by.csv: 26 rows
  columns: ['influence_type', 'sailor_work_id', 'influencing_work_id', 'key', 'sailor_work_name', 'sailor_work_year', 'sailor_work_genre', 'sailor_work_notable', 'influencing_work_name', 'influencing_work_year', 'influencing_work_genre', 'influencing_creators', 'influencing_creator_ids']

sailor_influenced_by_long.csv: 122 rows
  columns: ['influence_type', 'sailor_work_id', 'influencing_work_id', 'key', 'sailor_work_name', 'sailor_work_year', 'sailor_work_genre', 'sailor_work_notable', 'influencing_work_name', 'influencing_work_year', 'influencing_work_genre', 'creator_id', 'creator_name', 'creator_role']

sailor_influenced_others.csv: 0 rows
  columns: ['influence_type', 'key', 'sailor_work_id', 'sailor_work_name', 'sailor_work_year', 'sailor_work_genre', 'influenced_work_name', 'influenced_work_year', 'influenced_work_genre